In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)


In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
from openbabel import pybel
import shutil
import time
import pandas as pd
from rdkit import Chem

In [ ]:
mol_excel_file = "HTQC_reaction_thermo.xlsx"
df = pd.read_excel(mol_excel_file)

In [ ]:

MAX_TASKS = 2

In [ ]:

NPROC = 10
MEM = 20  

In [ ]:

os.makedirs('energy/success', exist_ok=True)
os.makedirs('energy/failure', exist_ok=True)

In [ ]:

opt_freq_path = "opt+freq/success"
energy_path = "energy"
success_dir = os.path.join(energy_path, "success")
failure_dir = os.path.join(energy_path, "failure")

In [ ]:
NEW_GAUSSIAN_KEYWORDS_gas = "M062X/6-311+G(2d,p) em=gd3"
NEW_GAUSSIAN_KEYWORDS_sol = "M062X/6-311+G(2d,p) em=gd3 scrf=(SMD,solvent=generic,read)"

In [ ]:

def convert_out_to_gjf(out_file, gjf_file):
    
    subprocess.run(["obabel", "-i", "out", out_file, "-o", "gjf", "-O", gjf_file])

In [ ]:

def extract_charge_and_coordinates(gjf_file):
    with open(gjf_file, 'r') as file:
        lines = file.readlines()
    
    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')
    
    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index - 1
            break
    
    
    final_index = None
    for index, line in enumerate(lines[start_index + 1 : ]):
        if atom_line_pattern.match(line) == False:
            final_index = index
            break
    
    
    charge_and_multiplicity_line = lines[start_index]
    coordinates_lines = lines[start_index + 1 : final_index]        
    
    return charge_and_multiplicity_line, coordinates_lines

In [ ]:

def modify_and_append_to_gjf(gjf_file, new_keywords, chk_directory, charge_and_coords, eps):
    current_dir = os.getcwd()
    chk_file = os.path.join(current_dir, gjf_file.replace('.gjf', '.chk'))
    with open(gjf_file, 'w') as file:
        file.write(f"%nprocshared={NPROC}\n")
        file.write(f"%mem={MEM}GB\n")
        file.write(f'%chk={chk_file}\n')
        file.write(f'# {new_keywords}\n\n')
        file.write(f'{gjf_file} energy ECW\n\n')
        file.write(charge_and_coords[0])  
        file.writelines(charge_and_coords[1])  
        if eps == 0:
            file.write('\n\n')
        else:
            file.write(f"eps={eps}\n")
            file.write(f"epsinf=2\n")
            file.write('\n\n')

In [ ]:


def run_gaussian_and_categorize(gjf_file, success_dir, failure_dir, reaction_index_file):
    

    
    base_name = os.path.splitext(os.path.basename(gjf_file))[0]
    
    out_file = base_name + ".out"
    chk_file = base_name + ".chk"
    
    gjf_dir = os.path.dirname(gjf_file)

    
    out_file_path = os.path.join(gjf_dir, out_file)
    chk_file_path = os.path.join(gjf_dir, chk_file)

    
    try:
        
        subprocess.run(["g16", gjf_file, out_file], check=True)

        
        with open(out_file_path, 'r') as file:
            contents = file.read()
            if "Normal termination" in contents:
                
                dest_dir = os.path.join(success_dir, reaction_index_file)
                
                if not os.path.exists(dest_dir):
                    os.makedirs(dest_dir)
                
                os.rename(gjf_file, os.path.join(dest_dir, os.path.basename(gjf_file)))
                os.rename(out_file_path, os.path.join(dest_dir, out_file))
                if os.path.exists(chk_file_path):
                    os.rename(chk_file_path, os.path.join(dest_dir, chk_file))
                return out_file_path, True
            else:
                
                dest_dir = os.path.join(failure_dir, reaction_index_file)
                if not os.path.exists(dest_dir):
                    os.makedirs(dest_dir)
                os.rename(gjf_file, os.path.join(dest_dir, os.path.basename(gjf_file)))
                os.rename(out_file_path, os.path.join(dest_dir, out_file))
                if os.path.exists(chk_file_path):
                    os.rename(chk_file_path, os.path.join(dest_dir, chk_file))
    except subprocess.CalledProcessError:
        
        dest_dir = os.path.join(failure_dir, reaction_index_file)
        if not os.path.exists(dest_dir):
            os.makedirs(dest_dir)
        os.rename(gjf_file, os.path.join(dest_dir, os.path.basename(gjf_file)))
        if os.path.exists(out_file_path):
            os.rename(out_file_path, os.path.join(dest_dir, out_file))
        if os.path.exists(chk_file_path):
            os.rename(chk_file_path, os.path.join(dest_dir, chk_file))

    return None, False

In [ ]:

def process_df(df, opt_freq_path, energy_path):
    
    reactant_name_cols = [col for col in df.columns if col.startswith('Reactant_Name_')]
    product_name_cols = [col for col in df.columns if col.startswith('Product_Name_')]

    
    for index, row in df.iterrows():
        
        reactant_and_product_name_list = list(row[reactant_name_cols].values) + list(row[product_name_cols].values)

        
        eps = row['eps']

        
        reaction_index_file = str(index)
        reaction_file = os.path.join(energy_path, reaction_index_file)
        if not os.path.exists(reaction_file):
            os.makedirs(reaction_file)

        
        for molecule_name in reactant_and_product_name_list:
            
            out_file = f"{molecule_name}.out"
            source_file = os.path.join(opt_freq_path, out_file)

            
            if os.path.exists(source_file):
                
                target_file = os.path.join(reaction_file, out_file.replace(".out", ".gjf"))

                
                convert_out_to_gjf(source_file, target_file)

                
                charge_and_coords = extract_charge_and_coordinates(target_file)

                if eps == 0:
                    
                    modify_and_append_to_gjf(target_file, NEW_GAUSSIAN_KEYWORDS_gas, reaction_file, charge_and_coords, eps)
                else:
                    
                    modify_and_append_to_gjf(target_file, NEW_GAUSSIAN_KEYWORDS_sol, reaction_file, charge_and_coords, eps)
            else:
                
                print("Public message removed for release.")

In [ ]:

process_df(df, opt_freq_path, energy_path)

In [ ]:



def run_all_gaussian_tasks(energy_path, success_dir, failure_dir, MAX_TASKS):
    
    
    reaction_index_file_list = [name for name in os.listdir(energy_path)
                                if os.path.isdir(os.path.join(energy_path, name))]

    
    files_to_process = []

    
    for reaction_index_file in reaction_index_file_list:
        
        reaction_folder = os.path.join(energy_path, reaction_index_file)
        
        for f in os.listdir(reaction_folder):
            
            if f.endswith('.gjf'):
                
                energy_gjf_file_path = os.path.join(reaction_folder, f)
                
                files_to_process.append((energy_gjf_file_path, reaction_index_file))

    
    with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
        
        futures = []
        
        for energy_gjf_file_path, reaction_index_file in files_to_process:
            
            future = executor.submit(run_gaussian_and_categorize, energy_gjf_file_path,
                                     success_dir, failure_dir, reaction_index_file)
            
            futures.append(future)
        
        for future in as_completed(futures):
            try:
                result = future.result()
                
                print("Public message removed for release.")
            except Exception as e:
                
                print("Public message removed for release.")

In [ ]:
run_all_gaussian_tasks(energy_path, success_dir, failure_dir, MAX_TASKS)